# Session 5 · Packaged Meaning: Defaults, Clichés and Ideology
*Cultural Machines: An Introduction*
Based on Leif Weatherby, *Language Machines: Cultural AI and the End of Remainder Humanism* (University of Minnesota Press, 2025).

**In this session you will:** scan a model's defaults (the words, people and places it reaches for when you don't specify), measure the gender assumptions attached to jobs in the arts, test how differently it treats English and African languages, and hunt for the ready-made phrases that fill machine-written prose.

**Time:** about 90 minutes. Please use *Runtime → Change runtime type → T4 GPU*.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

Left to itself, a language model takes the most travelled road through language. Here we borrow the art critic Clement Greenberg's description of kitsch, **predigested form**, and apply it to machine-written text. Call the result **packaged semantics**: bundles of meaning that arrive pre-assembled. A funding bid "aims to foster dialogue". A city is "vibrant". An artist "explores themes of identity".

On this view, this is what **ideology** looks like in language: not a political slogan, but the *manner* in which things are habitually said, the channels meaning usually flows through. The striking claim is that language models make ideology **measurable** for the first time. Because the model has absorbed a vast portion of what gets written, its defaults are a kind of X-ray of our collective habits.

He also warns that this window may close. As these tools become polished commercial products, their defaults get smoothed over and harder to see. Using small, raw models, as we do here, keeps the X-ray visible.

Two things to hold at once today: these defaults are *ours* (the machine learned them from us), and they are *amplified* (the machine repeats them at enormous scale).

## Setup

In [ ]:
#@title Setup: load GPT-2 (about a minute)
import torch, pandas as pd, matplotlib.pyplot as plt, re
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

gtok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2"); gpt2.eval()

def default_scan(prompt, n=60, words=1, seed=0):
    """Let GPT-2 finish the prompt n times and count what comes next."""
    set_seed(seed)
    ids = gtok(prompt, return_tensors="pt")
    out = gpt2.generate(**ids, do_sample=True, max_new_tokens=6, num_return_sequences=n,
                        top_p=0.95, pad_token_id=gtok.eos_token_id)
    endings = []
    for seq in out:
        cont = gtok.decode(seq[ids["input_ids"].shape[1]:], skip_special_tokens=True)
        endings.append(" ".join(re.findall(r"[A-Za-z'-]+", cont)[:words]).lower())
    counts = Counter(endings).most_common(12)
    plt.figure(figsize=(8, 4))
    plt.barh([c[0] for c in counts][::-1], [c[1] for c in counts][::-1], color="teal")
    plt.title(f"How GPT-2 finishes: '{prompt} ...'  ({n} tries)"); plt.tight_layout(); plt.show()

def pronoun_lean(roles, template="The {} said that"):
    """For each role, compare the probability of ' he' vs ' she' coming next."""
    he, she = gtok.encode(" he")[0], gtok.encode(" she")[0]
    rows = []
    for r in roles:
        ids = gtok(template.format(r), return_tensors="pt")["input_ids"]
        with torch.no_grad():
            p = torch.softmax(gpt2(ids).logits[0, -1], dim=-1)
        rows.append((r, float(p[she] / (p[he] + p[she]) * 100)))
    df = pd.DataFrame(rows, columns=["role", "% she"]).sort_values("% she")
    plt.figure(figsize=(8, 0.4 * len(roles) + 1))
    plt.barh(df["role"], df["% she"], color=["#b5651d" if v < 50 else "#2e8b57" for v in df["% she"]])
    plt.axvline(50, color="grey", ls="--"); plt.xlim(0, 100)
    plt.xlabel("share of 'she' (vs 'he'), %"); plt.title(f"'{template}' → he or she?")
    plt.tight_layout(); plt.show()
    return df

print("GPT-2 ready.")

## Part 1 · The default scan

We give GPT-2 an unfinished sentence and let it complete it 60 times. The chart shows what it reaches for most often. You are looking at the most travelled roads.

In [ ]:
default_scan("The famous artist was a")

In [ ]:
default_scan("The museum in Africa was full of")

In [ ]:
# Your turn: try "The audience at the opera was", "The musician from Lagos played",
# "The gallery in London was full of", "The young poet lived in a" ...
default_scan("The audience at the opera was")

**Record:** what surprised you? What did you predict correctly? Compare pairs that differ by only one word (Africa / Europe, opera / concert). The *difference* between the two charts is where the ideology shows.

## Part 2 · Who does the job?

In English, "the curator said that ___" is often followed by *he* or *she*. The chart shows how the model leans for each role in the arts. Grey dashed line = even.

In [ ]:
pronoun_lean(["nurse", "curator", "painter", "drummer", "museum director", "dancer",
              "sculptor", "fashion designer", "poet", "film director", "art teacher",
              "sound engineer", "choreographer", "gallery owner", "novelist"])

**Discussion:** Here is an uncomfortable argument. Critics are right that these models reproduce bias. But if a model trained on real language came out perfectly unbiased, that would arguably prove it had *failed* to learn our language, which is full of such assumptions. What is the difference between a mirror and a megaphone? Which is this?

## Part 3 · The language tax

Models do not treat all languages equally. One sign is **tokenisation**: how many pieces a sentence gets chopped into. More pieces means the model has seen less of that language, understands it less well, and (on paid services) it can literally cost more to use.

The sentences below are simple greetings. **Please check and correct them**, and add sentences in languages your participants speak.

In [ ]:
#@title Setup: load a small multilingual chat model (takes 1–3 minutes the first time)
# A small, openly available chat model. It is far weaker than ChatGPT or Claude,
# which is useful: its habits and defaults are easier to see.
import torch, textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL).to(device=device, dtype=dtype)

def ask(prompt, system="You are a helpful assistant.", temperature=0.8,
        max_new_tokens=250, show=True):
    """Send one message to the chat model and return its reply."""
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": prompt}]
    text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = chat_tok(text, return_tensors="pt").to(device)
    out = chat_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=temperature, top_p=0.9,
                              pad_token_id=chat_tok.eos_token_id)
    reply = chat_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if show:
        for para in reply.split("\n"):
            print(textwrap.fill(para, 90) if para.strip() else "")
    return reply

print(f"Chat model ready (running on {device}).")

In [ ]:
sentences = {
    "English": "Welcome. My name is Ama.",
    "French":  "Bienvenue. Je m'appelle Ama.",
    "Swahili": "Karibu. Jina langu ni Ama.",
    "Twi":     "Akwaaba. Me din de Ama.",
    "Yoruba":  "Ẹ káàbọ̀. Orúkọ mi ni Ama.",
    # "Your language": "Type the same sentence here",
}

rows = []
for lang, s in sentences.items():
    rows.append((lang, len(gtok(s)["input_ids"]), len(chat_tok(s)["input_ids"])))
df = pd.DataFrame(rows, columns=["language", "GPT-2 (2019) tokens", "Qwen (2024) tokens"])
display(df)
df.set_index("language").plot.bar(figsize=(8, 4), rot=0, title="Same greeting, different cost")
plt.ylabel("number of tokens"); plt.show()

In [ ]:
# See exactly how a sentence is chopped up
s = sentences["Twi"]
print([gtok.decode([i]) for i in gtok(s)["input_ids"]])
print([chat_tok.decode([i]) for i in chat_tok(s)["input_ids"]])

Now ask the chat model the same thing in different languages and compare the quality of its answers.

In [ ]:
ask("In two sentences, what is highlife music?")
print("\n---")
ask("Kwa sentensi mbili, muziki wa highlife ni nini?")   # Swahili: please check and edit

Note that the language these models capture is heavily tilted towards **present-day English**. For a cultural sector working across many languages, this is not a side issue. It decides whose culture is fluent inside the machine and whose arrives broken into fragments.

## Part 4 · Packaged semantics: the cliché hunt

Now the chat model. We ask it for the same kind of text several times and look for the phrases that keep coming back. These are the **packages**: pre-assembled meaning.

In [ ]:
#@title Setup: tools for finding repeated phrases (instant)
import re
from collections import Counter

STOP = set("""a an the and or but of to in on at for with by from as is are was were be been it its this that
these those i you he she we they my your our their his her me us them not no so if then than""".split())

def common_phrases(texts, sizes=(2, 3, 4), min_texts=3, top=20):
    """Find phrases that appear in at least `min_texts` different texts."""
    seen = Counter()
    for t in texts:
        words = re.findall(r"[a-z']+", t.lower())
        grams = set()
        for n in sizes:
            for i in range(len(words) - n + 1):
                g = words[i:i+n]
                if all(w in STOP for w in g):
                    continue
                grams.add(" ".join(g))
        seen.update(grams)
    rows = [(g, c) for g, c in seen.most_common() if c >= min_texts]
    # hide short phrases that only ever appear inside a longer repeated phrase
    rows = [(g, c) for g, c in rows
            if not any(g != h and f" {g} " in f" {h} " and c == d for h, d in rows)][:top]
    if not rows:
        print("No phrase appeared in that many texts. Try lowering min_texts.")
    for g, c in rows:
        print(f"{c:>3} texts  ·  {g}")
    return rows

def word_counts(texts, top=25):
    words = [w for t in texts for w in re.findall(r"[a-z']+", t.lower()) if w not in STOP and len(w) > 2]
    return Counter(words).most_common(top)

print("Ready.")

In [ ]:
statements = []
for i in range(8):
    print(f"\n--- artist statement {i+1} ---")
    statements.append(ask("Write a three-sentence artist statement for a young painter.",
                          max_new_tokens=120))

In [ ]:
print("PHRASES THAT KEEP COMING BACK:\n")
common_phrases(statements, min_texts=3)
print("\nMOST USED WORDS:\n", word_counts(statements, top=20))

Try other genres from your working life. Good candidates: a press release for a festival, a museum's mission statement, a review of a concert, a description of a city for tourists.

In [ ]:
texts = [ask("Write a two-sentence description of a typical African city for a tourism brochure.",
             max_new_tokens=100, show=False) for _ in range(8)]
common_phrases(texts, min_texts=2)

In [ ]:
texts = [ask("Write a two-sentence description of a typical European city for a tourism brochure.",
             max_new_tokens=100, show=False) for _ in range(8)]
common_phrases(texts, min_texts=2)

### Worksheet: an ideology scan

Choose one topic you care about. Run matched prompts that differ in one detail (a place, a gender, an art form, a language). For each, note:
1. the recurring phrases and images;
2. what is *never* said;
3. which version sounds more "natural" to you, and why that might be.

This is the seed of the capstone project option, an **ideology scan report**.

## Discussion

1. This course borrows Greenberg's word *kitsch* for machine prose. Is that fair? Is the sector's own language (funding bids, artist statements) already packaged in this way?
2. If these defaults are an X-ray of our collective habits, who should be using that X-ray, and for what?
3. As tools get more polished, these defaults become harder to see. What might your organisation lose if it stops noticing them?

## Going further
- Bender, Gebru, McMillan-Major and Mitchell, "On the Dangers of Stochastic Parrots" (2021), the classic critical paper on bias.
- The Masakhane community's work on natural language processing for African languages.